#### Import Dependencies

In [47]:
from llama_index.core import SimpleDirectoryReader, Document
from llama_index.core.vector_stores import MetadataFilter
from llama_index.core.extractors import QuestionsAnsweredExtractor
from llama_index.core.storage.docstore import SimpleDocumentStore
from llama_index.core.node_parser import (
    HierarchicalNodeParser,
    get_child_nodes,
    get_root_nodes,
    get_leaf_nodes
)
from llama_index.llms.openai import OpenAI
from llama_index.core.ingestion import IngestionPipeline
from llama_index.core import VectorStoreIndex
from llama_index.vector_stores.weaviate import WeaviateVectorStore
from IPython.display import Markdown, display
from llama_index.core import StorageContext
from llama_index.embeddings.openai import OpenAIEmbedding
from llama_index.core.retrievers import AutoMergingRetriever

import re
from tqdm import tqdm
import os
import openai
import weaviate

# environment variable for OpenAI API key
import os
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
#### Data Loading
pdf_path = r"C:\Users\USER\Desktop\Projects\rag_llamaindex\tools\rag_docs\legal_docs\international law handook.pdf"
docs = SimpleDirectoryReader(input_files=[pdf_path]).load_data()

In [3]:
#### Chunk the data
chunks = HierarchicalNodeParser.from_defaults(
    chunk_sizes=[2000, 1000, 500],  
    chunk_overlap=50
)

### metafilters
qa_extractor = QuestionsAnsweredExtractor(questions=3)

In [6]:
#### Set Up LLM for embeddings
llm = OpenAI(model="gpt-4o") 
embed_model = OpenAIEmbedding(model="text-embedding-3-small") 


In [8]:
transformations = [
    chunks,
    qa_extractor
]

In [9]:
pipeline = IngestionPipeline(
    transformations=transformations
)
nodes = pipeline.run(documents=docs)
print(f"Total nodes created: {len(nodes)}")

  0%|          | 9/2720 [00:06<24:49,  1.82it/s]2026-02-08 22:37:37,610 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-02-08 22:37:37,710 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
  1%|          | 14/2720 [00:08<15:57,  2.83it/s]2026-02-08 22:37:38,936 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-02-08 22:37:39,286 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
  1%|          | 24/2720 [00:11<18:47,  2.39it/s]2026-02-08 22:37:42,574 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-02-08 22:37:42,879 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-02-08 22:37:42,973 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
  1%|          | 28/2720 [00:12<14:18,  3.14it/s]2026-02-08 22:

Total nodes created: 2720


In [28]:
first_node = nodes[0]
first_node.extra_info

C:\Users\USER\AppData\Local\Temp\ipykernel_14404\700810936.py:2: DeprecationWarning: Call to deprecated function (or staticmethod) extra_info. ('extra_info' is deprecated, use 'metadata' instead.) -- Deprecated since version 0.12.2.
  first_node.extra_info


{'page_label': 'i',
 'file_name': 'international law handook.pdf',
 'file_path': 'C:\\Users\\USER\\Desktop\\Projects\\rag_llamaindex\\tools\\rag_docs\\legal_docs\\international law handook.pdf',
 'file_type': 'application/pdf',
 'file_size': 6209766,
 'creation_date': '2026-01-23',
 'last_modified_date': '2026-01-23',
 'questions_this_excerpt_can_answer': '1. What is the title of the international law handbook and what is its content focused on?\n2. What is the file path and file size of the international law handbook PDF document?\n3. When was the international law handbook last modified and created?'}

In [48]:
client = weaviate.connect_to_weaviate_cloud(
    cluster_url=os.getenv("WEAVIATE_URL"),
    auth_credentials=weaviate.auth.AuthApiKey(os.getenv("WEAVIATE_API_KEY"))
)

class_name = "InternationalLawDocument"

if client.collections.exists(class_name):
    client.collections.delete(class_name)
    print("Deleted existing collection")

vector_store = WeaviateVectorStore(
    weaviate_client=client,
    index_name=class_name,
    text_key="content",
)

docstore = SimpleDocumentStore()
# insert nodes into docstore
docstore.add_documents(nodes)

storage_context = StorageContext.from_defaults(
    vector_store=vector_store,
    docstore=docstore,)

# Create OpenAI embedding model
embedding_model = OpenAIEmbedding(api_key=os.getenv("OPENAI_API_KEY"), model="text-embedding-3-small")

base_index = VectorStoreIndex(
    nodes,
    storage_context=storage_context,
    embedding=embedding_model,
)

2026-02-09 00:35:25,228 - INFO - HTTP Request: GET https://uwtnr9odtaymiq4qso8fea.c0.us-east1.gcp.weaviate.cloud/v1/meta "HTTP/1.1 200 OK"
2026-02-09 00:35:26,624 - INFO - HTTP Request: GET https://pypi.org/pypi/weaviate-client/json "HTTP/1.1 200 OK"
2026-02-09 00:35:27,208 - INFO - HTTP Request: GET https://uwtnr9odtaymiq4qso8fea.c0.us-east1.gcp.weaviate.cloud/v1/schema/InternationalLawDocument "HTTP/1.1 200 OK"
2026-02-09 00:35:27,679 - INFO - HTTP Request: DELETE https://uwtnr9odtaymiq4qso8fea.c0.us-east1.gcp.weaviate.cloud/v1/schema/InternationalLawDocument "HTTP/1.1 200 OK"


Deleted existing collection


2026-02-09 00:35:27,948 - INFO - HTTP Request: GET https://uwtnr9odtaymiq4qso8fea.c0.us-east1.gcp.weaviate.cloud/v1/schema/InternationalLawDocument "HTTP/1.1 404 Not Found"
2026-02-09 00:35:28,338 - INFO - HTTP Request: POST https://uwtnr9odtaymiq4qso8fea.c0.us-east1.gcp.weaviate.cloud/v1/schema "HTTP/1.1 200 OK"
c:\Users\USER\Desktop\Projects\rag_llamaindex\.venv\Lib\site-packages\weaviate\warnings.py:302: ResourceWarning: Con004: The connection to Weaviate was not closed properly. This can lead to memory leaks.
            Please make sure to close the connection using `client.close()`.
  warnings.warn(
C:\Users\USER\AppData\Local\Temp\ipykernel_14404\1376528886.py:12: ResourceWarning: unclosed <ssl.SSLSocket fd=6000, family=2, type=1, proto=0, laddr=('192.168.100.122', 62556), raddr=('35.201.124.182', 443)>
  vector_store = WeaviateVectorStore(
2026-02-09 00:35:32,537 - INFO - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
2026-02-09 00:35:35,848 - INFO - 

In [49]:
print(nodes[0].embedding)
# print(len(obj.vector))

None


In [56]:
# Create the AutoMergingRetriever
base_retriever = base_index.as_retriever(similarity_top_k=6)
retriever = AutoMergingRetriever(
    base_retriever, 
    storage_context, 
    verbose=True,
    )

# Retrieve results
qn = "What are the purposes of the United Nations?"

results_nodes = retriever.retrieve(qn)
base_nodes = base_retriever.retrieve(qn)

# Filter by similarity score > 0.5
filtered_nodes = [node for node in base_nodes if node.score > 0.5]

for i, result in enumerate(filtered_nodes):
    print(f"--- Base Node {i+1} ---")
    print("Content (stored node):", result.node.get_content())
    print("Similarity Score:", result.score)
    if hasattr(result.node, "extra_info"):
        print("Metadata:", result.node.extra_info)
    print("\n")

2026-02-09 00:55:42,288 - INFO - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
2026-02-09 00:55:43,942 - INFO - > Merging 1 nodes into parent node.
> Parent node id: db8aa16f-4b0d-43c5-a312-cfea4669810a.
> Parent node text: 3
1. Charter o f t he UnI ted n at Ions
done at san f rancisco on 26 June 1945
entry into force: ...

2026-02-09 00:55:43,943 - INFO - > Merging 1 nodes into parent node.
> Parent node id: c9af400c-143f-4623-9500-d4b2f323fb8f.
> Parent node text: 327
30. unIt Ing for P eaCe
general assembly resolution 377 (V) of 3 november 1950
a
The General ...



> Merging 1 nodes into parent node.
> Parent node id: db8aa16f-4b0d-43c5-a312-cfea4669810a.
> Parent node text: 3
1. Charter o f t he UnI ted n at Ions
done at san f rancisco on 26 June 1945
entry into force: ...

> Merging 1 nodes into parent node.
> Parent node id: c9af400c-143f-4623-9500-d4b2f323fb8f.
> Parent node text: 327
30. unIt Ing for P eaCe
general assembly resolution 377 (V) of 3 november 1950
a
The General ...



2026-02-09 00:55:44,456 - INFO - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"


--- Base Node 1 ---
Content (stored node): 3
1. Charter o f t he UnI ted n at Ions
done at san f rancisco on 26 June 1945
entry into force: 24 october 1945
We the peoples of the United Nations
determined to save succeeding generations from the scourge of war, which twice in our life -
time has brought untold sorrow to mankind, and 
to reaffirm faith in fundamental human rights, in the dignity and worth of the human person, 
in the equal rights of men and women and of nations large and small, and
to establish conditions under which justice and respect for the obligations arising from treaties 
and other sources of international law can be maintained, and 
to promote social progress and better standards of life in larger freedom, 
and for these ends
to practice tolerance and live together in peace with one another as good neighbours, and 
to unite our strength to maintain international peace and security, and 
to ensure, by the acceptance of principles and the institution of methods, tha

C:\Users\USER\AppData\Local\Temp\ipykernel_14404\2949071038.py:22: DeprecationWarning: Call to deprecated function (or staticmethod) extra_info. ('extra_info' is deprecated, use 'metadata' instead.) -- Deprecated since version 0.12.2.
  if hasattr(result.node, "extra_info"):
C:\Users\USER\AppData\Local\Temp\ipykernel_14404\2949071038.py:23: DeprecationWarning: Call to deprecated function (or staticmethod) extra_info. ('extra_info' is deprecated, use 'metadata' instead.) -- Deprecated since version 0.12.2.
  print("Metadata:", result.node.extra_info)
